In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 新数据 - 计算时间已修正
data = {
    '组合': [
        "确定性 + FNN + 无物理约束",
        "确定性 + FNN + 有物理约束",
        "确定性 + Transformer + 无物理约束",
        "确定性 + Transformer + 有物理约束",
        "α-VI + FNN + 无物理约束",
        "α-VI + FNN + 有物理约束",
        "α-VI + Transformer + 无物理约束",
        "α-VI + Transformer + 有物理约束"
    ],
    '贝叶斯方法': ['确定性', '确定性', '确定性', '确定性', 'α-VI', 'α-VI', 'α-VI', 'α-VI'],
    '架构': ['FNN', 'FNN', 'Transformer', 'Transformer', 'FNN', 'FNN', 'Transformer', 'Transformer'],
    '物理约束': ['无', '有', '无', '有', '无', '有', '无', '有'],
    'Params': [5129, 5129, 27913, 27913, 10258, 10258, 55570, 55570],
    'Time(s)': [68.25, 75.83, 96.87, 119.94, 1045.19, 1318.66, 1728.63, 2189.57],
    'RelL2': [0.058, 0.031, 0.035, 0.021, 0.047, 0.026, 0.023, 0.014],
    'MSE': [0.001408, 0.000853, 0.000977, 0.000234, 0.001004, 0.000474, 0.000328, 0.000086]
}

df = pd.DataFrame(data)

print("=" * 70)
print("新数据概览（计算时间已修正）")
print("=" * 70)
print(df[['组合', 'Params', 'Time(s)', 'MSE']].to_string(index=False))

print("\n" + "=" * 70)
print("主效应分析（基于MSE - 越小越好）")
print("=" * 70)

# 主效应计算
factors = ['贝叶斯方法', '架构', '物理约束']
for factor in factors:
    effect = df.groupby(factor)['MSE'].mean()
    improvement = (1 - effect.min() / effect.max()) * 100
    print(f"\n{factor}:")
    for idx, val in effect.items():
        print(f"  {idx}: {val:.6f}")
    print(f"  → 改进幅度: {improvement:.1f}%")

# 时间成本分析
print("\n" + "=" * 70)
print("计算时间成本分析")
print("=" * 70)
time_ratio = df.groupby('贝叶斯方法')['Time(s)'].mean()
print(f"确定性平均时间: {time_ratio['确定性']:.1f}s")
print(f"α-VI平均时间: {time_ratio['α-VI']:.1f}s")
print(f"时间倍数: {time_ratio['α-VI']/time_ratio['确定性']:.1f}x")

In [ ]:
# 交互效应分析（新数据）
print("交互效应分析（新数据）")
print("=" * 70)

# 1. 贝叶斯 × 架构
ba = df.groupby(['贝叶斯方法', '架构'])['MSE'].mean().unstack()
print("\n贝叶斯方法 × 架构:")
print(ba)
det_trans_imp = (1 - ba.loc['确定性', 'Transformer']/ba.loc['确定性', 'FNN'])*100
alpha_trans_imp = (1 - ba.loc['α-VI', 'Transformer']/ba.loc['α-VI', 'FNN'])*100
print(f"Transformer提升(确定性): {det_trans_imp:.1f}%")
print(f"Transformer提升(α-VI): {alpha_trans_imp:.1f}%")
print(f"协同效应: {alpha_trans_imp - det_trans_imp:+.1f}%")

# 2. 架构 × 约束
ac = df.groupby(['架构', '物理约束'])['MSE'].mean().unstack()
print("\n架构 × 物理约束:")
print(ac)
fnn_const_imp = (1 - ac.loc['FNN', '有']/ac.loc['FNN', '无'])*100
trans_const_imp = (1 - ac.loc['Transformer', '有']/ac.loc['Transformer', '无'])*100
print(f"约束提升(FNN): {fnn_const_imp:.1f}%")
print(f"约束提升(Transformer): {trans_const_imp:.1f}%")
print(f"协同效应: {trans_const_imp - fnn_const_imp:+.1f}%")

# 性价比分析（精度提升 / 计算时间）
df['MSE_improvement'] = (df['MSE'].max() - df['MSE']) / df['MSE'].max() * 100
df['Cost_Efficiency'] = df['MSE_improvement'] / df['Time(s)']

print("\n" + "=" * 70)
print("性价比分析（精度提升率 / 计算时间）")
print("=" * 70)
efficiency_df = df[['组合', '贝叶斯方法', '架构', '物理约束', 'Time(s)', 'MSE', 'Cost_Efficiency']].sort_values('Cost_Efficiency', ascending=False)
print(efficiency_df.to_string(index=False))

# 帕累托最优分析
print("\n" + "=" * 70)
print("帕累托最优分析（精度 vs 时间）")
print("=" * 70)
pareto = []
for i, row in df.iterrows():
    is_dominated = False
    for j, other in df.iterrows():
        if i != j:
            # 如果other在时间和精度上都更好或相等，且至少一个严格更好
            if other['Time(s)'] <= row['Time(s)'] and other['MSE'] <= row['MSE']:
                if other['Time(s)'] < row['Time(s)'] or other['MSE'] < row['MSE']:
                    is_dominated = True
                    break
    if not is_dominated:
        pareto.append(row)

pareto_df = pd.DataFrame(pareto).sort_values('Time(s)')
print("\n帕累托前沿（无法被其他配置同时超越时间和精度）：")
print(pareto_df[['组合', 'Time(s)', 'MSE']].to_string(index=False))

In [ ]:
# 新数据可视化
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('三因素实验分析（修正后数据）', fontsize=18, fontweight='bold', y=0.98)

# 颜色定义
colors = {'确定性': '#3498db', 'α-VI': '#e74c3c'}
marker_style = {'FNN': 'o', 'Transformer': 's'}

# 1. 分组柱状图 - 精度对比
ax = axes[0, 0]
x = np.arange(4)
width = 0.35
configs = ['FNN\n无约束', 'FNN\n有约束', 'Trans\n无约束', 'Trans\n有约束']

det_mse = df[df['贝叶斯方法']=='确定性']['MSE'].values
alpha_mse = df[df['贝叶斯方法']=='α-VI']['MSE'].values

bars1 = ax.bar(x - width/2, det_mse, width, label='确定性', color=colors['确定性'], alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, alpha_mse, width, label='α-VI', color=colors['α-VI'], alpha=0.8, edgecolor='black')

ax.set_ylabel('MSE (对数尺度)', fontsize=12, fontweight='bold')
ax.set_title('(a) 精度对比：贝叶斯方法 × 架构 × 物理约束', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(configs, fontsize=11)
ax.legend(fontsize=11)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height, f'{height:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height, f'{height:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. 时间-精度帕累托图（关键图）
ax = axes[0, 1]
for method in ['确定性', 'α-VI']:
    subset = df[df['贝叶斯方法'] == method]
    for arch in ['FNN', 'Transformer']:
        arch_data = subset[subset['架构'] == arch]
        marker = marker_style[arch]
        ax.scatter(arch_data['Time(s)'], arch_data['MSE'],
                  s=arch_data['Params']/100,
                  c=colors[method], marker=marker,
                  alpha=0.7, edgecolors='black', linewidth=1.5,
                  label=f'{method}+{arch}')

# 帕累托前沿线
pareto_times = [68.25, 75.83, 119.94, 2189.57]
pareto_mses = [0.001408, 0.000853, 0.000234, 0.000086]
ax.plot(pareto_times, pareto_mses, 'k--', linewidth=2, alpha=0.6, label='帕累托前沿')

ax.set_xlabel('计算时间 (s)', fontsize=12, fontweight='bold')
ax.set_ylabel('MSE', fontsize=12, fontweight='bold')
ax.set_title('(b) 帕累托前沿分析\n(气泡大小=参数量)', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=9, loc='upper right')

# 标注帕累托点
pareto_labels = ['基线', '+约束', '+Trans', '+α-VI']
for t, m, label in zip(pareto_times, pareto_mses, pareto_labels):
    ax.annotate(label, (t, m), xytext=(10, 10), textcoords='offset points',
                fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# 3. 交互效应热力图
ax = axes[1, 0]
# 创建架构×约束的交互表
pivot_mse = df.pivot_table(values='MSE', index=['架构', '贝叶斯方法'], columns='物理约束', aggfunc='mean')
im = ax.imshow(pivot_mse.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(pivot_mse.columns)))
ax.set_yticks(range(len(pivot_mse.index)))
ax.set_xticklabels(pivot_mse.columns, fontsize=11)
ax.set_yticklabels([f"{idx[0]}\n{idx[1]}" for idx in pivot_mse.index], fontsize=10)
ax.set_title('(c) MSE热力图\n(颜色越绿越好)', fontsize=13, fontweight='bold')

# 添加数值标注
for i in range(len(pivot_mse.index)):
    for j in range(len(pivot_mse.columns)):
        val = pivot_mse.values[i, j]
        text = ax.text(j, i, f'{val:.6f}', ha="center", va="center",
                      color="white" if val < 0.0005 else "black", fontweight='bold', fontsize=9)
plt.colorbar(im, ax=ax, label='MSE')

# 4. 性价比与边际改进
ax = axes[1, 1]
df_sorted = df.sort_values('Cost_Efficiency', ascending=True)
y_pos = np.arange(len(df_sorted))

# 创建渐变色
colors_bar = [colors[m] for m in df_sorted['贝叶斯方法']]
bars = ax.barh(y_pos, df_sorted['Cost_Efficiency'], color=colors_bar, alpha=0.7, edgecolor='black')

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['贝叶斯方法']}+{row['架构']}+{row['物理约束']}"
                    for _, row in df_sorted.iterrows()], fontsize=10)
ax.set_xlabel('性价比 (精度提升率/秒)', fontsize=12, fontweight='bold')
ax.set_title('(d) 性价比排序\n(越高越好)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3, linestyle='--')

# 添加数值标签
for i, (bar, val) in enumerate(zip(bars, df_sorted['Cost_Efficiency'])):
    ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('/mnt/kimi/output/updated_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n图表已保存至: /mnt/kimi/output/updated_analysis.png")

In [ ]:
# 补充：边际效应图（Interaction Plot）- 更清晰展示交互
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('边际效应分析：因素间的协同与拮抗作用', fontsize=16, fontweight='bold')

# 1. 物理约束的边际效应（按架构分层）
ax = axes[0]
for arch in ['FNN', 'Transformer']:
    subset = df[df['架构'] == arch].sort_values('物理约束')
    ax.plot(subset['物理约束'], subset['MSE'], marker='o', linewidth=3, markersize=10,
            label=f'{arch}', alpha=0.8)

    # 添加改进幅度标注
    no_const = subset[subset['物理约束']=='无']['MSE'].values[0]
    yes_const = subset[subset['物理约束']=='有']['MSE'].values[0]
    improvement = (1 - yes_const/no_const) * 100
    mid_x = 0.5
    mid_y = (no_const + yes_const) / 2
    ax.annotate(f'↓{improvement:.0f}%', xy=(mid_x, mid_y), fontsize=11, fontweight='bold',
                ha='center', color='darkgreen' if arch=='Transformer' else 'darkblue',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_ylabel('MSE', fontsize=12, fontweight='bold')
ax.set_title('(a) 物理约束 × 架构\n(线条斜率越大，协同越强)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# 2. Transformer的边际效应（按贝叶斯方法分层）
ax = axes[1]
for method in ['确定性', 'α-VI']:
    subset = df[df['贝叶斯方法'] == method].sort_values('架构')
    ax.plot(subset['架构'], subset['MSE'], marker='s', linewidth=3, markersize=10,
            label=f'{method}', alpha=0.8)

    # 添加改进幅度标注
    fnn = subset[subset['架构']=='FNN']['MSE'].values[0]
    trans = subset[subset['架构']=='Transformer']['MSE'].values[0]
    improvement = (1 - trans/fnn) * 100
    mid_x = 0.5
    mid_y = (fnn + trans) / 2
    ax.annotate(f'↓{improvement:.0f}%', xy=(mid_x, mid_y), fontsize=11, fontweight='bold',
                ha='center', color='darkred' if method=='α-VI' else 'darkblue',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_ylabel('MSE', fontsize=12, fontweight='bold')
ax.set_title('(b) 架构 × 贝叶斯方法\n(α-VI下Transformer收益更大)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('/mnt/kimi/output/marginal_effects.png', dpi=150, bbox_inches='tight')
plt.show()

# 输出关键结论
print("\n" + "="*70)
print("关键发现总结（基于修正后数据）")
print("="*70)
print(f"1. 最强协同效应：物理约束 + Transformer = 75.5%精度提升")
print(f"   (vs FNN+约束仅45%，协同增益 +30.5%)")
print(f"\n2. 架构选择 × 贝叶斯方法协同：")
print(f"   - 确定性下Transformer提升: 46.4%")
print(f"   - α-VI下Transformer提升: 72.0%")
print(f"   - 协同增益: +25.6%")
print(f"\n3. 性价比之王：确定性 + Transformer + 物理约束 (0.695)")
print(f"   (精度提升83.4%，仅需120秒)")
print(f"\n4. 帕累托前沿4个配置构成最优选择集：")
print(f"   - 快速便宜: 确定性+FNN+无 (68s)")
print(f"   - 平衡: 确定性+FNN+有 (76s)")
print(f"   - 高精度/低成本: 确定性+Trans+有 (120s) ← 推荐")
print(f"   - 极致精度: α-VI+Trans+有 (2190s)")
print("="*70)